# Dehergne repertoire analysis

> First time use: follow instructions in the [README.md](README.md) file in this directory.

## Setup

Create a TimelinkNotebook object. 

Note:
* First time run takes a little time as the required Docker images are downloaded.
* Timelink will default to using sqlite as the database, see [Receipts notebook for more control](1-receipts.ipynb)


In [ ]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook(kleio_image="kleio-server", kleio_version="12.9.592")
tlnb.print_info(show_token=True)

### Database status

Count the number of rows in each table in the database.


In [ ]:
tlnb.table_row_count_df()

### Check the status of the files

Check the import status of the translated files

* I: Imported
* E: Imported with error
* W: Imported with warnings no errors
* N: Not imported
* U: Translation updated need to reimport

In [ ]:
kleio_files = tlnb.get_kleio_files()
# kleio_files.info()
kleio_files[["name","import_status","status","errors","warnings","import_errors","import_warnings"]]

## Save ids of Jesuits from a given place of entry

In [ ]:
import pandas as pd
from timelink.pandas import entities_with_attribute
from dehergne_util import calc_age_at
# show 500 rows
pd.set_option('display.max_rows', 1550)

place_of_entry = "Coimbra"

jesuits_of_place = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname','extra_info'],
    the_type='jesuita-entrada',
    the_value=place_of_entry,
    more_attributes=['dehergne','nascimento','embarque','morte'],
    db=tlnb.db,
)
place_of_entry_ids = jesuits_of_place.index.tolist()


In [ ]:
place_of_stay=None# set to None to show all places

## List all attributes related to places along with place of entry and birth

In [ ]:
import pandas as pd
from timelink.pandas import entities_with_attribute
# show 500 rows
pd.set_option('display.max_rows', 150)

ids = place_of_entry_ids

places_of_stay = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname','the_line','the_order','extra_info'],
    the_type=['jesuita-entrada','partida','chegada','estadia','estadia-x',
                'jesuita-votos-local','jesuita-ordenacao-padre','morte'],
    the_value=place_of_stay,
    column_name='place',
    more_attributes=['jesuita-entrada','nascimento'],
    # filter_by=ids,
    db=tlnb.db,
)

# fillna with "?"
places_of_stay.place = places_of_stay.place.fillna('?')

# remove rows where place == '?'
places_of_stay = places_of_stay[places_of_stay.place != '?']

# this sequence replaces missing dates with the value of the previous date
# followed by ">". This allows to use the information that although the date
# is unkown it has happened after a certain date
# we need the id in a column to group by it (it is currently in the index)
# TODO: maybe an option in entities_with_attribute and compute intervals using
#       post quem and ante quem dates. See https://github.com/time-link/timelink-kleio/issues/1

places_of_stay['id_col'] = places_of_stay.index

# We try to infer the date of stay from the previous date known
# create a copy of the date column to replace the '0' values with NaN
places_of_stay['place.date_inferred'] = places_of_stay['place.date'].replace('0', pd.NA)

# order by id and line to have the missing date values filled with the previous date
places_of_stay = places_of_stay.sort_values(by=['id_col', 'place.line'])
# use ffill to fill the missing values with the previous value
places_of_stay['place.date_inferred'] = places_of_stay.groupby(['id_col'])['place.date_inferred'].ffill()
# create a column to flag the inferred dates
places_of_stay['place.date_is_inferred'] = places_of_stay['place.date_inferred'] != places_of_stay['place.date']
# reset index
places_of_stay.reset_index(inplace=True)
# if data_is_inferred set date_inferred to date_inferrred+">"
places_of_stay.loc[places_of_stay['place.date_is_inferred'], 'place.date_inferred'] = places_of_stay['place.date_inferred'] + '>'
# set missing value to ''
places_of_stay['place.date_inferred'] = places_of_stay['place.date_inferred'].fillna('')
# restore index
places_of_stay.index = places_of_stay['id_col']

places_of_stay.info()


### Extract the wikidata_id from the comments of place names

In [ ]:
import pandas as pd
import re

def extract_wikidata_id(extra_info_dict):
    """
    Extract wikidata ID from place.extra_info JSON string.
    Returns the wikidata ID (e.g., 'Q6625') or NaN if not present.
    """
    if pd.isna(extra_info_dict):
        return pd.NA
    
    try:
        # Try to parse as JSON first (standard format with double quotes)
        
        comment = extra_info_dict.get('the_value', {}).get('comment', '')
        if comment and 'wikidata:' in comment:
            # Extract wikidata ID using regex pattern (Q followed by digits)
            match = re.search(r'wikidata:(Q\d+)', comment)
            if match:
                return match.group(1)
        return pd.NA
    except (ValueError, SyntaxError, TypeError):
        
        return pd.NA


In [ ]:

# Apply the function to create the new column
places_of_stay['place.wikidata_id'] = places_of_stay['place.extra_info'].apply(extract_wikidata_id)

In [ ]:

# show results
show_only=10
cols=['place','place.wikidata_id','place.date_inferred','groupname','name','place.type', 'nascimento', 'nascimento.date', 'place.type','jesuita-entrada', 'jesuita-entrada.date', 'place.date_is_inferred','place.line','place.extra_info']
places_of_stay[places_of_stay.groupname=='n'][cols].sort_values(by=['place','place.date_inferred','name',]).head(show_only)



### Save to Excel

In [ ]:
places_of_stay[cols].sort_values(by=['place','place.date_inferred']).to_excel(f"../inferences/jesuita_entrada_places_of_stay.xlsx", sheet_name='wikidata', index=True)

### List unlocated places (no wikidata)

In [ ]:
show_only=10
places_of_stay[(places_of_stay['place.wikidata_id'].isna())][cols].sort_values(by=['place','place.date_inferred']).head(show_only)

#### Export to Excel all unlocated places

In [ ]:
places_of_stay[(places_of_stay['place.wikidata_id'].isna())][cols].sort_values(by=['place','place.date_inferred']).to_excel(f"../inferences/jesuita_entrada_places_of_stay_no_wikidata.xlsx", sheet_name='no_wikidata', index=True)

#### Only specific place of entry, unkown places

In [ ]:
places_of_stay_specific = places_of_stay.loc[place_of_entry_ids]
places_of_stay_specific[(places_of_stay_specific['place.wikidata_id'].isna())][cols].sort_values(by=['place','place.date_inferred'])

In [ ]:
places_of_stay_specific[~(places_of_stay_specific['place.wikidata_id'].isna())][cols].sort_values(by=['place','place.date_inferred']).to_excel(f"../inferences/jesuita_entrada_{place_of_entry}_places_of_stay_no_wikidata.xlsx", sheet_name='no_wikidata', index=True)

## Frequency of places / Frequência dos lugares

In [ ]:
places_of_stay.info()

In [ ]:
col = 'place' # subtotal by this column

# Use pandas groupby and specify unique value count for id
df_totals = places_of_stay.groupby(col).agg({'id':'nunique',
                                                'place.date_inferred':'min',
                                                'place.date':'max'})
df_totals.info()
df_totals.sort_values('id',ascending= False).head(130)

## Priesthood / Ordens sacras

Those that received the sacrament of priesthood in a given place

In [ ]:
import pyuca  # to sort accented characters properly

import pandas as pd
from timelink.pandas import entities_with_attribute
# show 500 rows
pd.set_option('display.max_rows', 550)

jesuita_ordenacao_padre = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname'],
    the_type='jesuita-ordenacao-padre',
    the_value='Coimbra',
    more_attributes=['dehergne','jesuita-entrada','jesuita-votos-local','nascimento', ],
    db=tlnb.db,
)
jesuita_ordenacao_padre.info()

coll = pyuca.Collator()

# we use the Collator class to sort the names and countries properly
# since Python does not sort accented characters properly
# we create two new columns with the sort keys

# first we fill the NaN values with empty strings
jesuita_ordenacao_padre['jesuita-votos-local'] = jesuita_ordenacao_padre['jesuita-votos-local'].fillna('?')
jesuita_ordenacao_padre['jesuita-votos-local'] = jesuita_ordenacao_padre['jesuita-votos-local'].fillna('?')

# now we add two columns with sort keys
jesuita_ordenacao_padre['jesuita-votos-loc-sort'] = jesuita_ordenacao_padre['jesuita-votos-local'].apply(coll.sort_key)
jesuita_ordenacao_padre['name_sort'] = jesuita_ordenacao_padre['name'].apply(coll.sort_key)
jesuita_ordenacao_padre

## Vows / Votos

In [ ]:
import pyuca  # to sort accented characters properly

import pandas as pd
from timelink.pandas import entities_with_attribute
# show 500 rows
pd.set_option('display.max_rows', 550)

jesuita_entrada_full = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname'],
    the_type='jesuita-votos',
    more_attributes=['jesuita-votos-local','nascimento', 'jesuita-entrada'],
    db=tlnb.db,
)
jesuita_entrada_full.info()

coll = pyuca.Collator()

# we use the Collator class to sort the names and countries properly
# since Python does not sort accented characters properly
# we create two new columns with the sort keys

# first we fill the NaN values with empty strings
jesuita_entrada_full['jesuita-votos-local'] = jesuita_entrada_full['jesuita-votos-local'].fillna('?')
jesuita_entrada_full['jesuita-votos-local'] = jesuita_entrada_full['jesuita-votos-local'].fillna('?')

# now we add two columns with sort keys
jesuita_entrada_full['jesuita-votos-loc-sort'] = jesuita_entrada_full['jesuita-votos-local'].apply(coll.sort_key)
jesuita_entrada_full['name_sort'] = jesuita_entrada_full['name'].apply(coll.sort_key)


In [ ]:
# filter dataframe with rows with index in place_of_entry_ids


In [ ]:
from timelink.kleio.utilities import convert_timelink_date

df = jesuita_entrada_full[jesuita_entrada_full.index.isin(place_of_entry_ids)].copy()
# calculate age at date "jesuita-votos.date" considering "nascimento.date" as date of birth
# and using the function convert_timelink_date to convert the Timelink date to a Python datetime
df['nascimento.date_2'] = df['nascimento.date'].fillna('0').apply(convert_timelink_date)
df['jesuita-votos.date_2'] = df['jesuita-votos.date'].fillna('0').apply(convert_timelink_date)
# calculate diference in years between the two dates
df['age_at_votos'] = df.apply(lambda row: calc_age_at(row['nascimento.date_2'], row['jesuita-votos.date_2']), axis=1)
df['age_at_votos'] = df['age_at_votos'].fillna(0).astype(int)
df[['nascimento.date','jesuita-votos.date','jesuita-votos','age_at_votos']].head(50)


## Voyages


In [ ]:
import pandas as pd
from timelink.pandas import entities_with_attribute
# show 500 rows
pd.set_option('display.max_rows', 550)

voyage_of_interest = 25

voyage = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname'],
    the_type='wicky-viagem',
    the_value=str(voyage_of_interest),
    more_attributes=['embarque','chegada','partida',],
    db=tlnb.db,
)
voyage

## Visualizar registos / View records





### Visualizar uma pessoa / View a person


#### Atributos de uma pessoa numa tabela, uma linha por attributo

---

#### Person attributes in a dataframe, one line per attribute

In [ ]:
import pandas as pd
from timelink.pandas import group_attributes as person_attributes

pd.set_option('display.max_rows',1000)

id = 'deh-bento-de-gois'
pdf = person_attributes([id],db=tlnb.db)  # note id in a list
pdf[['the_date','the_type','the_value','attr_obs']].sort_values(['the_date','the_type'])

In [ ]:
tlnb.db.pperson(id)


#### Atributos de uma pessoa numa tabela, attributos em colunas

---

#### Person attributes in a dataframe, attributes in columns

In [ ]:
# Get list of people with with a certain value in a specific attribute

id = 'deh-andre-palmeiro'  # João Pedro Ribeiro
# id = '234295'  # Alexandre Metelo de Sousa
df = entities_with_attribute(
                    entity_type='person',
                    the_type='jesuita-entrada',  # we need a base attribute
                    show_elements=['name'],
                    more_attributes=['jesuita-votos','jesuita-votos-local'],
                    db=tlnb.db,
                    filter_by=[id],
                    sql_echo=False)
view_cols = ['name','jesuita-entrada','jesuita-entrada.date','jesuita-votos','jesuita-votos-local']
df.info()
df[view_cols].sort_values('jesuita-entrada.date')

# Examinar potenciais duplicados

---

## Examine potentital duplicates

In [ ]:
from timelink.pandas import display_group_attributes
pd.set_option('display.max_rows',250)

no_show=['código-de-referência','data-do-registo','url','faculdade.ano','naturalidade.ano',
         'matricula-faculdade.ano','nome-apelido','nome-primeiro','nome-geografico.ano',
         'grau.ano','matricula-outra.ano','nome-geografico','instituta.ano']

dup_ids = ['deh-joao-barradas-ref1','deh-lazzaro-cattaneo-ref1','deh-matteo-ricci'] 

display_group_attributes(dup_ids,
                             header_attributes=['jesuita-entrada','nacionalidade'],
                             exclude_attributes=no_show,
                             sort_attributes=['the_date','the_type','the_value'],
                             cmap_name='Pastel1',
                             db=tlnb.db,)


In [ ]:
# print current timestamp
from datetime import datetime

# print current timestamp
print(f"Analysis completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")